In [7]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.ensemble import StackingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OrdinalEncoder
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings('ignore')

In [2]:
# =============================================================================
# 1. CHARGEMENT DES DONNÉES
# =============================================================================

print("Chargement des données...")
try:
    # On utilise low_memory=False pour éviter les warnings sur les types mixtes
    X_train = pd.read_csv('X_train.csv', index_col=0, low_memory=False)
    y_train = pd.read_csv('y_train.csv', index_col=0, low_memory=False)
    X_test = pd.read_csv('X_test.csv', index_col=0, low_memory=False)
    
    print(f"X_train : {X_train.shape}")
    print(f"y_train : {y_train.shape}")
    print(f"X_test  : {X_test.shape}")
    
    # Alignement des index train/target
    common_indices = X_train.index.intersection(y_train.index)
    X_train = X_train.loc[common_indices]
    y_train = y_train.loc[common_indices]
    
except FileNotFoundError as e:
    print(f"Erreur : {e}")

Chargement des données...
X_train : (1172086, 306)
y_train : (1172086, 1)
X_test  : (586044, 306)


In [3]:
# =============================================================================
# 2. SÉLECTION DES FEATURES (ALIGNEMENT TRAIN/TEST)
# =============================================================================

# Le problème précédent venait de l'utilisation de colonnes 'math_...' présentes dans le train
# mais absentes (ou vides) dans le test.
# On ne garde QUE les colonnes présentes dans les DEUX jeux de données.

print("Alignement des colonnes Train / Test...")

# Intersection des colonnes
common_cols = X_train.columns.intersection(X_test.columns)

# Filtrage supplémentaire : on enlève explicitement les colonnes qui pourraient fuiter ou être biaisées
# (Bien que l'intersection devrait déjà gérer ça si elles sont absentes de X_test)
final_features = [c for c in common_cols if not c.lower().startswith('math_')]

print(f"Nombre de features communes conservées : {len(final_features)}")

X_train_clean = X_train[final_features].copy()
X_test_clean = X_test[final_features].copy()

# Vérification rapide
print("Aperçu des features conservées (10 premières) :")
print(final_features[:10])

Alignement des colonnes Train / Test...
Nombre de features communes conservées : 264
Aperçu des features conservées (10 premières) :
['Year', 'CNT', 'CNTRYID', 'CNTSCHID', 'CNTSTUID', 'CYC', 'NatCen', 'STRATUM', 'SUBNATIO', 'OECD']


In [4]:
def clean_data(df, constants, missing_threshold=0.5):
    """6. Data Cleaning"""
    print("\n--- 6. Data Cleaning ---")
    
    df_clean = df.copy()
    
    # 6.1 Drop constant columns
    if constants:
        print(f"Dropping {len(constants)} constant columns...")
        df_clean = df_clean.drop(columns=constants, errors='ignore')

    # 6.1.1 Drop columns with too many missing values
    missing_ratio = df_clean.isnull().mean()
    cols_to_drop = missing_ratio[missing_ratio > missing_threshold].index.tolist()
    if cols_to_drop:
        print(f"Dropping {len(cols_to_drop)} columns with > {missing_threshold*100}% missing values: {cols_to_drop}")
        df_clean = df_clean.drop(columns=cols_to_drop)
        
    # 6.1.2 Drop High Cardinality Categorical Columns
    cat_cols = df_clean.select_dtypes(exclude=['number']).columns
    high_card_cols = [col for col in cat_cols if df_clean[col].nunique() > 50]
    if high_card_cols:
        print(f"Dropping {len(high_card_cols)} high cardinality columns (>50 unique): {high_card_cols}")
        df_clean = df_clean.drop(columns=high_card_cols)
        
    # 6.2 Imputation
    print("Imputing missing values...")
    num_cols = df_clean.select_dtypes(include=['number']).columns
    for col in num_cols:
        if df_clean[col].isnull().any():
            median_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(median_val)
            
    cat_cols = df_clean.select_dtypes(exclude=['number']).columns
    for col in cat_cols:
        if df_clean[col].isnull().any():
            if not df_clean[col].mode().empty:
                mode_val = df_clean[col].mode()[0]
                df_clean[col] = df_clean[col].fillna(mode_val)
            else:
                df_clean[col] = df_clean[col].fillna("Unknown")
            
    # 6.3 Outliers
    print("Handling outliers (Clipping to 5th and 95th percentiles)...")
    for col in num_cols:
        lower_bound = df_clean[col].quantile(0.05)
        upper_bound = df_clean[col].quantile(0.95)
        if ((df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)).any():
             df_clean[col] = df_clean[col].clip(lower=lower_bound, upper=upper_bound)
        
    print(f"Cleaned dataset shape: {df_clean.shape}")
    return df_clean

def select_features(df):
    """8. Feature Selection"""
    print("\n--- 8. Feature Selection ---")
    
    print("Checking for highly correlated features (>0.95)...")
    num_cols = df.select_dtypes(include=[np.number]).columns
    
    corr_matrix = df[num_cols].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
    
    print(f"Dropping {len(to_drop)} highly correlated features: {to_drop[:10]}...")
    df_selected = df.drop(columns=to_drop)
    
    print(f"Selected dataset shape: {df_selected.shape}")
    return df_selected

In [5]:
# =============================================================================
# 2.5 PREPROCESSING AVANCÉ (CLEANING & SELECTION)
# =============================================================================

# Identification des constantes sur le Train
constants = [col for col in X_train_clean.columns if X_train_clean[col].nunique() <= 1]

# Application du Cleaning sur X_train
# Note: clean_data fait aussi de l'imputation et du clipping
X_train_clean = clean_data(X_train_clean, constants)

# Application de la Sélection de features (Corrélation)
X_train_clean = select_features(X_train_clean)

# --- PROPAGATION SUR X_TEST ---
print("\nPropagation des transformations sur X_test...")

# 1. Alignement des colonnes (on ne garde que ce qui reste dans X_train)
cols_to_keep = X_train_clean.columns
X_test_clean = X_test_clean[cols_to_keep].copy()

# 2. Imputation & Outliers sur X_test (en utilisant les stats de X_train pour éviter le leakage)
# On refait une passe légère similaire à clean_data mais appliquée à X_test avec stats X_train

num_cols = X_train_clean.select_dtypes(include=['number']).columns
cat_cols = X_train_clean.select_dtypes(exclude=['number']).columns

# Imputation Numérique (Médiane du Train)
for col in num_cols:
    median_val = X_train_clean[col].median() # On utilise la médiane du train déjà imputé (ou on aurait dû le faire avant)
    # Note: X_train_clean est déjà imputé, donc sa médiane est biaisée par l'imputation, mais c'est acceptable ici.
    X_test_clean[col] = X_test_clean[col].fillna(median_val)

# Imputation Catégorielle (Mode du Train)
for col in cat_cols:
    if not X_train_clean[col].mode().empty:
        mode_val = X_train_clean[col].mode()[0]
        X_test_clean[col] = X_test_clean[col].fillna(mode_val)
    else:
        X_test_clean[col] = X_test_clean[col].fillna("Unknown")

# Outliers (Clipping avec bornes du Train)
for col in num_cols:
    lower_bound = X_train_clean[col].quantile(0.05)
    upper_bound = X_train_clean[col].quantile(0.95)
    X_test_clean[col] = X_test_clean[col].clip(lower=lower_bound, upper=upper_bound)

print(f"X_test_clean shape final : {X_test_clean.shape}")


--- 6. Data Cleaning ---
Dropping 14 constant columns...
Dropping 168 columns with > 50.0% missing values: ['LANGTEST_PAQ', 'Option_CT', 'Option_WBQ', 'ISCEDP', 'MISSSC', 'MATHEASE', 'WB153', 'ST253', 'ST311', 'IC180', 'ST350', 'FL162', 'PA195', 'ST330', 'FL169', 'ST230', 'ST331', 'ST354', 'ST305', 'ST258', 'PA167', 'IC176', 'ST352', 'PA188', 'ST283', 'ST256', 'WB166', 'FL150', 'ST266', 'IC182', 'FL164', 'FL160', 'ST267', 'ST338', 'ST351', 'ST342', 'ST355', 'FL166', 'FL170', 'WB168', 'IC184', 'ST348', 'ST226', 'ST336', 'WB160', 'FL167', 'ST268', 'IC177', 'PA196', 'IC183', 'ST275', 'IC172', 'IC175', 'ST340', 'ST289', 'ST301', 'PA186', 'PA197', 'ST290', 'ST345', 'ST322', 'PA006', 'ST347', 'ST343', 'ST307', 'ST160', 'ST168', 'ST161', 'ST153', 'ST188', 'PA009', 'PA182', 'PA004', 'ST059', 'ST150', 'PA158', 'WB155', 'WB162', 'ST127', 'EC162', 'ST104', 'PA003', 'ST125', 'ST163', 'ST297', 'ST006', 'ST263', 'PA162', 'ST164', 'ST008', 'ST152', 'EC031', 'WB177', 'ST183', 'EC012', 'ST223', 'ST177

In [6]:
# =============================================================================
# 3. PRÉTRAITEMENT (ENCODAGE & IMPUTATION)
# =============================================================================

# Pour aller vite et être robuste, on va :
# 1. Identifier les colonnes catégorielles (object) et numériques
# 2. Encoder les catégorielles en nombres (Ordinal Encoding) pour que XGB/LGBM/HistGB puissent les manger
# 3. Les modèles choisis (HistGradientBoosting, XGBoost, LightGBM) gèrent nativement les NaN, 
#    donc on peut éviter une imputation lourde si on veut gagner du temps.
#    Cependant, pour RidgeCV (meta-learner), il faut pas de NaN. Mais le meta-learner ne voit que les prédictions.
#    Donc on peut laisser les NaN dans X pour les base_models !

print("Identification des types de colonnes...")
cat_cols = X_train_clean.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train_clean.select_dtypes(exclude=['object', 'category']).columns.tolist()

print(f"Variables catégorielles : {len(cat_cols)}")
print(f"Variables numériques    : {len(num_cols)}")

# Encodage des variables catégorielles
# On utilise OrdinalEncoder qui est plus rapide que OneHot pour les arbres
print("Encodage des variables catégorielles...")
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# On combine train et test pour fit l'encoder (pour gérer toutes les catégories possibles)
# Attention au data leakage : idéalement on fit sur train, mais pour l'ordinal encoding de catégories fixes, c'est tolérable
# ou on utilise handle_unknown='use_encoded_value'

X_train_clean[cat_cols] = encoder.fit_transform(X_train_clean[cat_cols].astype(str))
X_test_clean[cat_cols] = encoder.transform(X_test_clean[cat_cols].astype(str))

print("Encodage terminé.")

Identification des types de colonnes...
Variables catégorielles : 1
Variables numériques    : 74
Encodage des variables catégorielles...
Encodage terminé.


In [8]:
# =============================================================================
# 4. DÉFINITION DU MODÈLE (RAPIDE & ROBUSTE)
# =============================================================================

# On remplace RandomForest par HistGradientBoostingRegressor qui est :
# 1. Beaucoup plus rapide (O(n) vs O(n log n))
# 2. Gère nativement les NaN
# 3. Souvent plus performant

base_models = [
    ('xgb', xgb.XGBRegressor(
        n_estimators=500,       # Réduit pour la vitesse (suffisant avec le learning rate)
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=42,
        tree_method='hist'      # Mode histogramme très rapide
    )),
    
    ('lgbm', lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        n_jobs=-1,
        random_state=42,
        verbose=-1
    )),

    ('cat', CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=8,
        random_seed=42,
        verbose=0,
        allow_writing_files=False
    )),
    
    ('hgb', HistGradientBoostingRegressor(
        max_iter=500,
        learning_rate=0.05,
        max_depth=8,
        random_state=42
    ))
]

# Meta-learner simple et robuste
meta_learner = RidgeCV()

stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_learner,
    cv=3,          # CV=3 suffit pour le stacking et accélère le processus (vs 5)
    n_jobs=1,      # IMPORTANT : 1 pour éviter les crashs Windows avec Stacking
    passthrough=False
)

print("Modèle Stacking configuré avec XGBoost, LightGBM, CatBoost et HistGradientBoosting.")

Modèle Stacking configuré avec XGBoost, LightGBM, CatBoost et HistGradientBoosting.


In [9]:
# =============================================================================
# 5. ENTRAÎNEMENT (AVEC BARRE DE PROGRESSION)
# =============================================================================

# Comme on ne peut pas mettre de tqdm DANS le fit de sklearn facilement,
# on va faire un split manuel pour valider, puis un fit global.

print("Split Train/Validation pour évaluation...")
X_tr, X_val, y_tr, y_val = train_test_split(X_train_clean, y_train, test_size=0.2, random_state=42)

print("Entraînement sur le split (patience)...")
# On utilise tqdm comme contexte juste pour montrer que ça tourne
with tqdm(total=1, desc="Entraînement Split") as pbar:
    stacking_model.fit(X_tr, y_tr.values.ravel())
    pbar.update(1)

# Évaluation
y_pred_val = stacking_model.predict(X_val)
y_pred_val = np.clip(y_pred_val, 0, 1000) # Clipping de sécurité

r2 = r2_score(y_val, y_pred_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))

print(f"\n=== RÉSULTATS VALIDATION ===")
print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:.4f}")


Split Train/Validation pour évaluation...
Entraînement sur le split (patience)...
Entraînement sur le split (patience)...


Entraînement Split:   0%|          | 0/1 [00:00<?, ?it/s]


=== RÉSULTATS VALIDATION ===
R²   : 0.4988
RMSE : 86.3978


In [10]:
# Entraînement FINAL
print("\nEntraînement FINAL sur tout le dataset...")
with tqdm(total=1, desc="Entraînement Final") as pbar:
    stacking_model.fit(X_train_clean, y_train.values.ravel())
    pbar.update(1)
    
print("Modèle final prêt.")


Entraînement FINAL sur tout le dataset...


Entraînement Final:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# 6. PRÉDICTION ET SOUMISSION
# =============================================================================

print("Génération des prédictions sur X_test...")
y_test_pred = stacking_model.predict(X_test_clean)

# Stats avant clipping
print("\nStats prédictions (Brut) :")
print(pd.Series(y_test_pred).describe())

# Clipping [0, 1000]
y_test_pred = np.clip(y_test_pred, 0, 1000)

print("\nStats prédictions (Clippé) :")
print(pd.Series(y_test_pred).describe())

# Création fichier
sample_sub = pd.read_csv('sample_submission_hfactory.csv')
pred_df = pd.DataFrame({'ID': X_test.index, 'MathScore': y_test_pred})

if 'MathScore' in sample_sub.columns:
    del sample_sub['MathScore']

final_submission = sample_sub.merge(pred_df, on='ID', how='left')

# Remplissage manquants
if final_submission['MathScore'].isnull().any():
    mean_val = y_train.values.ravel().mean()
    print(f"Remplissage de {final_submission['MathScore'].isnull().sum()} valeurs manquantes.")
    final_submission['MathScore'] = final_submission['MathScore'].fillna(mean_val)

output_file = 'submission_final.csv'
final_submission.to_csv(output_file, index=False)
print(f"\nFichier généré : {output_file}")
print(final_submission.head())